# TP - Teoria de Compiladores
### Grupo: 4

###Integrantes
- ** Diego Fabrizio Mucha Alvarez - u202317069 **
- ** Nicole Yessenia Vasquez Tinco - u202322884**
- ** Alessandro Daniel Bravo Castillo - u202224501**
- ** - **
- ** - **

In [1]:
!pip install antlr4-python3-runtime==4.13.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 4.4 MB/s eta 0:00:00


In [2]:
!curl -O https://www.antlr.org/download/antlr-4.13.2-complete.jar

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2089k  100 2089k    0     0  2148k      0 --:--:-- --:--:-- --:--:-- 2147k


In [11]:
%%writefile Formas.g4
grammar Formas;

programa: instrucciones EOF;

instrucciones: instruccion*;

instruccion
    : asignacion
    | punto
    | recta
    | triangulo
    ;

asignacion: ID IGUAL expr # VAsignacion;
punto: PUNTO ID LPAREN expr COMA expr RPAREN # VPunto;
recta: RECTA ID LPAREN expr COMA expr COMA expr COMA expr RPAREN # VRecta;
triangulo: TRIANGULO ID LPAREN expr COMA expr COMA expr COMA expr COMA expr COMA expr RPAREN # VTriangulo;


expr
    : NUM # VValor
    | ID # VID
    ;

PUNTO      : 'punto';
RECTA      : 'recta';
TRIANGULO  : 'triangulo';

IGUAL : '=';
COMA  : ',';
LPAREN: '(';
RPAREN: ')';

ID  : [a-zA-Z][a-zA-Z0-9]*;
NUM : [0-9]+ ('.' [0-9]+)?;

WS : [ \t\r\n]+ -> skip;

Overwriting Formas.g4


In [4]:
!java -jar antlr-4.13.2-complete.jar -Dlanguage=Python3 -visitor Formas.g4

In [5]:
%%writefile Visitor.py
if __name__ is not None and "." in __name__:
    from .FormasParser import FormasParser
    from .FormasVisitor import FormasVisitor
else:
    from FormasParser import FormasParser
    from FormasVisitor import FormasVisitor

class Visitor(FormasVisitor):
    html = """ """
    rectas = {}

    def __init__(self):
      self.variables = {}

    def getRectas(self):
      return self.rectas

    def visitVAsignacion(self, ctx):
      l = list(ctx.getChildren())
      self.variables[l[0].getText()] = int(l[2].getText())

    def visitPrograma(self,ctx):
      print("hola")
      for i in list(ctx.getChildren()):
          self.visit(i)

    def visitInstrucciones(self,ctx):
      print("hola 2")
      for i in list(ctx.getChildren()):
          self.visit(i)

    def visitVRecta(self, ctx):
      l = list(ctx.getChildren())
      # Ver puntos
      print("PUNTOS: ")
      id = l[1].getText()
      self.rectas[id] = [l[3].getText(),l[5].getText(),l[7].getText(),l[9].getText()]


    def visitVValor(self,ctx):
      return int(ctx.NUM().getText())

    def visitVID(self,ctx):
      return int(ctx.NUM().getText())


Writing Visitor.py


In [6]:
%%writefile ejemplo1.xyz

recta B (100,100,200,200)

Writing ejemplo1.xyz


In [7]:
from antlr4 import *
from FormasLexer import FormasLexer
from FormasParser import FormasParser
from FormasVisitor import FormasVisitor

from Visitor import Visitor
import sys

input_stream = FileStream('ejemplo1.xyz')

lexer = FormasLexer(input_stream)
token_stream = CommonTokenStream(lexer)
parser = FormasParser(token_stream)

tree = parser.programa()
print(tree.toStringTree(recog=parser))

visitor = Visitor()
visitor.visit(tree)
print(visitor.getRectas())

(programa (instrucciones (instruccion (recta recta B ( (expr 100) , (expr 100) , (expr 200) , (expr 200) )))) <EOF>)
hola
hola 2
PUNTOS: 
{'B': ['100', '100', '200', '200']}


In [9]:
from IPython.display import HTML


html_content = """
<div style="padding: 20px; background-color: #f0f0f0; border-radius: 10px;">
  <h1 style="color: #4285F4;"> Interfaz de figuras</h1>
  <canvas id="canvas" width="500" height="500"></canvas>
    <script>
      function draw() {
        const canvas = document.getElementById("canvas");
        const ctx = canvas.getContext("2d");
        ctx.font = "10px Arial";

        ctx.beginPath();
        ctx.moveTo(100, 100);
        ctx.lineTo(200, 200);
        ctx.stroke();
        ctx.fillText("(100, 100)",70, 110);
        ctx.fillText("(200, 100)",170, 210);
      }
      draw();
    </script>

</div>


"""


HTML(html_content)